In [8]:
import numpy as np
import xarray as xr

# inputs you have:
# center_lat, center_lon   - launch pad / scene center (deg)
# u100, v100               - ERA5 100m wind (m/s, eastward / northward)
# orbit                    - e.g. "SSO", "ISS", "LEO"
# delta_t_secs             - launch-to-overpass lag (s)

def offset(lat, lon, bearing_deg, dist_km):
    dN = dist_km * np.cos(np.radians(bearing_deg)) / 111.32
    dE = dist_km * np.sin(np.radians(bearing_deg)) / (111.32 * np.cos(np.radians(lat)))
    return lat + dN, lon + dE

def launch_azimuth(inclination_deg, lat_deg):
    s = np.cos(np.radians(inclination_deg)) / np.cos(np.radians(lat_deg))
    beta = np.degrees(np.arcsin(np.clip(s, -1, 1)))
    return beta % 360, (180 - beta) % 360   # (prograde/easterly, retrograde/southerly)

def plume_path(center_lat, center_lon, u100, v100, delta_t_secs,
               inclination_deg, retrograde=False, downrange_km=None):
    # 1. launch azimuth from inclination
    pro, retro = launch_azimuth(inclination_deg, center_lat)
    az = retro if retrograde else pro

    # 2. wind: magnitude + bearing TOWARD (atan2(u,v) handles direction)
    spd = np.hypot(u100, v100)
    wind_dir = np.degrees(np.arctan2(u100, v100)) % 360
    drift_km = spd * delta_t_secs / 1000.0

    # 3. release points along trajectory (downrange per altitude); pad if None
    if downrange_km is None:
        downrange_km = np.array([0.0])
    downrange_km = np.atleast_1d(downrange_km)

    path = []
    for dr in downrange_km:
        rlat, rlon = offset(center_lat, center_lon, az, dr)         # release
        plat, plon = offset(rlat, rlon, wind_dir, drift_km)         # advected
        path.append((rlat, rlon, plat, plon))
    return az, wind_dir, drift_km, path


center_lat = 28.562
center_lon = -80.5772
ds = xr.open_dataset("/home/ucfacb0/ARIA/aria_data_analysis/data/era5/era5_CC_LC40_20230103_1500.nc")
u100 = ds["u100"].isel(valid_time=2,latitude=0,longitude=0)
v100 = ds["u100"].isel(valid_time=2,latitude=0,longitude=0)
delta_t_secs = 3225.592

az, wdir, drift, path = plume_path(
    center_lat, center_lon, u100, v100, delta_t_secs,
    inclination_deg=97.5, retrograde=True,
    downrange_km=[0, 1, 3, 5])     # from your gravity-turn model

for rlat, rlon, plat, plon in path:
    print(f"release {rlat:.4f},{rlon:.4f} -> plume {plat:.4f},{plon:.4f}")

release 28.5620,-80.5772 -> plume 28.4964,-80.6518
release 28.5531,-80.5787 -> plume 28.4876,-80.6533
release 28.5353,-80.5818 -> plume 28.4698,-80.6564
release 28.5176,-80.5848 -> plume 28.4520,-80.6594
